In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 200)

In [3]:
pd.describe_option()


compute.use_bottleneck : bool
    Use the bottleneck library to accelerate if it is installed,
    the default is True
    Valid values: False,True
    [default: True] [currently: True]
compute.use_numba : bool
    Use the numba engine option for select operations if it is installed,
    the default is False
    Valid values: False,True
    [default: False] [currently: False]
compute.use_numexpr : bool
    Use the numexpr library to accelerate computation if it is installed,
    the default is True
    Valid values: False,True
    [default: True] [currently: True]
display.chop_threshold : float or None
    if set to a float value, all float values smaller than the given threshold
    will be displayed as exactly 0 by repr and friends.
    [default: None] [currently: None]
display.colheader_justify : 'left'/'right'
    Controls the justification of column headers. used by DataFrameFormatter.
    [default: right] [currently: right]
display.date_dayfirst : boolean
    When True, prints an

In [4]:
File = "Project.csv"
df_raw = pd.read_csv(File)
print(df_raw.shape)
df_raw.head()

(2000, 11)


,Shipment_ID,Origin_Warehouse,Destination,Carrier,Shipment_Date,Delivery_Date,Weight_kg,Cost,Status,Distance_miles,Transit_Days
0,SH10000,Warehouse_MIA,San Francisco,UPS,10/2/2023,10/4/2023,25.7,67.46,Delivered,291,2
1,SH10001,Warehouse_MIA,Atlanta,DHL,12/6/2023,12/9/2023,38.9,268.85,Delivered,1225,3
2,SH10002,Warehouse_LA,Houston,DHL,9/18/2023,9/20/2023,37.2,74.35,Delivered,220,2
3,SH10003,Warehouse_BOS,Seattle,OnTrac,1/26/2023,2/4/2023,42.6,187.04,Delivered,1156,9
4,SH10004,Warehouse_SF,Dallas,OnTrac,6/3/2023,6/6/2023,7.9,120.01,Delivered,1017,3


In [5]:
df_raw.shape
df_raw.head()



,Shipment_ID,Origin_Warehouse,Destination,Carrier,Shipment_Date,Delivery_Date,Weight_kg,Cost,Status,Distance_miles,Transit_Days
0,SH10000,Warehouse_MIA,San Francisco,UPS,10/2/2023,10/4/2023,25.7,67.46,Delivered,291,2
1,SH10001,Warehouse_MIA,Atlanta,DHL,12/6/2023,12/9/2023,38.9,268.85,Delivered,1225,3
2,SH10002,Warehouse_LA,Houston,DHL,9/18/2023,9/20/2023,37.2,74.35,Delivered,220,2
3,SH10003,Warehouse_BOS,Seattle,OnTrac,1/26/2023,2/4/2023,42.6,187.04,Delivered,1156,9
4,SH10004,Warehouse_SF,Dallas,OnTrac,6/3/2023,6/6/2023,7.9,120.01,Delivered,1017,3


In [6]:
df_raw.dtypes

Shipment_ID          object
Origin_Warehouse     object
Destination          object
Carrier              object
Shipment_Date        object
Delivery_Date        object
Weight_kg           float64
Cost                float64
Status               object
Distance_miles        int64
Transit_Days          int64
dtype: object

In [7]:
df_raw["Shipment_ID"].dtype

dtype('O')

In [8]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Shipment_ID       2000 non-null   object 
 1   Origin_Warehouse  2000 non-null   object 
 2   Destination       2000 non-null   object 
 3   Carrier           2000 non-null   object 
 4   Shipment_Date     2000 non-null   object 
 5   Delivery_Date     1968 non-null   object 
 6   Weight_kg         2000 non-null   float64
 7   Cost              1959 non-null   float64
 8   Status            2000 non-null   object 
 9   Distance_miles    2000 non-null   int64  
 10  Transit_Days      2000 non-null   int64  
dtypes: float64(2), int64(2), object(7)
memory usage: 172.0+ KB


In [9]:
df_raw[["Shipment_Date", "Delivery_Date"]].dtypes

Shipment_Date    object
Delivery_Date    object
dtype: object

In [10]:
df_raw["Shipment_ID"] = df_raw["Shipment_ID"].astype(str)

In [11]:
df_raw["Shipment_ID"].dtype

dtype('O')

In [12]:
df_raw["Shipment_Date"] = pd.to_datetime(df_raw["Shipment_Date"], errors="coerce")

In [13]:
df_raw["Delivery_Date"] = pd.to_datetime(df_raw["Delivery_Date"], errors="coerce")

In [14]:
df_raw["Shipment_Date"].dtype
df_raw["Delivery_Date"].dtype

dtype('<M8[ns]')

In [15]:
df_raw["Delivery_Date"].isna().sum()

np.int64(32)

In [16]:
df_raw["Transit_Days"].isna().sum()

np.int64(0)

In [17]:
df_raw[df_raw["Delivery_Date"].isna()]["Status"].value_counts()

Status
Delivered    27
Delayed       5
Name: count, dtype: int64

In [18]:
df_raw[df_raw["Status"] == "Delivered"]["Delivery_Date"].isna().sum()

np.int64(27)

In [19]:
(df_raw["Delivery_Date"] - df_raw["Shipment_Date"]).dt.days.describe()

count    1968.000000
mean        4.618394
std         4.370629
min       -30.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        20.000000
dtype: float64

In [20]:
df_raw["Transit_Days"].describe()

count    2000.000000
mean        4.182500
std         1.837902
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        12.000000
Name: Transit_Days, dtype: float64

In [21]:
bad_mask = (df_raw["Delivery_Date"] - df_raw["Shipment_Date"]).dt.days < 0

In [22]:
bad_mask.sum()

np.int64(16)

In [23]:
df_raw.loc[bad_mask, "Delivery_Date"] = (
    df_raw.loc[bad_mask, "Shipment_Date"] +
    pd.to_timedelta(df_raw.loc[bad_mask, "Transit_Days"], unit="D")
)

In [24]:
(df_raw["Delivery_Date"] - df_raw["Shipment_Date"]).dt.days

0       2.0
1       3.0
2       2.0
3       9.0
4       3.0
       ... 
1995    5.0
1996    5.0
1997    6.0
1998    5.0
1999    NaN
Length: 2000, dtype: float64

In [25]:
diff = (df_raw["Delivery_Date"] - df_raw["Shipment_Date"]).dt.days - df_raw["Transit_Days"]

In [26]:
diff.describe()

count    1968.000000
mean        0.707825
std         2.439403
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        13.000000
dtype: float64

In [27]:
df_raw["Estimated_Delivery_Date"] = (
    df_raw["Shipment_Date"] +
    pd.to_timedelta(df_raw["Transit_Days"], unit="D")
)

In [28]:
df_raw.shape

(2000, 12)

In [29]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Shipment_ID              2000 non-null   object        
 1   Origin_Warehouse         2000 non-null   object        
 2   Destination              2000 non-null   object        
 3   Carrier                  2000 non-null   object        
 4   Shipment_Date            2000 non-null   datetime64[ns]
 5   Delivery_Date            1968 non-null   datetime64[ns]
 6   Weight_kg                2000 non-null   float64       
 7   Cost                     1959 non-null   float64       
 8   Status                   2000 non-null   object        
 9   Distance_miles           2000 non-null   int64         
 10  Transit_Days             2000 non-null   int64         
 11  Estimated_Delivery_Date  2000 non-null   datetime64[ns]
dtypes: datetime64[ns](3), float64(2), 

In [30]:
df_raw

,Shipment_ID,Origin_Warehouse,Destination,Carrier,Shipment_Date,Delivery_Date,Weight_kg,Cost,Status,Distance_miles,Transit_Days,Estimated_Delivery_Date
0,SH10000,Warehouse_MIA,San Francisco,UPS,2023-10-02,2023-10-04,25.7,67.46,Delivered,291,2,2023-10-04
1,SH10001,Warehouse_MIA,Atlanta,DHL,2023-12-06,2023-12-09,38.9,268.85,Delivered,1225,3,2023-12-09
2,SH10002,Warehouse_LA,Houston,DHL,2023-09-18,2023-09-20,37.2,74.35,Delivered,220,2,2023-09-20
3,SH10003,Warehouse_BOS,Seattle,OnTrac,2023-01-26,2023-02-04,42.6,187.04,Delivered,1156,9,2023-02-04
4,SH10004,Warehouse_SF,Dallas,OnTrac,2023-06-03,2023-06-06,7.9,120.01,Delivered,1017,3,2023-06-06
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,SH11995,Warehouse_BOS,San Francisco,FedEx,2023-01-18,2023-01-23,7.9,217.78,Delivered,1616,5,2023-01-23
1996,SH11996,Warehouse_HOU,Phoenix,UPS,2023-10-27,2023-11-01,36.5,279.47,Delivered,1708,5,2023-11-01
1997,SH11997,Warehouse_HOU,Portland,LaserShip,2023-02-13,2023-02-19,11.4,250.32,Delivered,1912,6,2023-02-19
1998,SH11998,Warehouse_SEA,Detroit,USPS,2023-10-17,2023-10-22,10.9,272.31,Delivered,2188,5,2023-10-22


In [31]:
df_raw["Cost"].isnull().value_counts()

Cost
False    1959
True       41
Name: count, dtype: int64

In [32]:
df_raw[df_raw["Cost"].isna()]["Status"].value_counts()

Status
Delivered    36
Delayed       5
Name: count, dtype: int64

In [33]:
df_raw[["Cost", "Distance_miles", "Weight_kg"]].corr()

,Cost,Distance_miles,Weight_kg
Cost,1.000000,0.435586,0.0212
Distance_miles,0.435586,1.000000,-0.0097
Weight_kg,0.021200,-0.009700,1.0000


In [34]:
df_raw.groupby("Carrier")["Cost"].mean()

Carrier
Amazon Logistics    193.909251
DHL                 222.134764
FedEx               217.965952
LaserShip           207.857905
OnTrac              184.599017
UPS                 229.714542
USPS                182.661032
Name: Cost, dtype: float64

In [35]:
df_raw[df_raw["Cost"].isna()]["Carrier"].value_counts()

Carrier
USPS                11
LaserShip            7
Amazon Logistics     7
DHL                  6
UPS                  5
OnTrac               4
FedEx                1
Name: count, dtype: int64

In [36]:
df_raw["Cost_per_Mile"] = df_raw["Cost"] / df_raw["Distance_miles"]

In [37]:
carrier_cpm = df_raw.groupby("Carrier")["Cost_per_Mile"].mean()

In [38]:
df_raw.groupby("Carrier")["Cost_per_Mile"].describe()

,count,mean,std,min,25%,50%,75%,max
Carrier,,,,,,,,
Amazon Logistics,267.0,0.171177,0.088489,0.114025,0.132619,0.147827,0.166607,0.888333
DHL,275.0,0.187965,0.068654,0.016644,0.152993,0.169636,0.201124,0.869141
FedEx,294.0,0.185665,0.105694,0.014061,0.146001,0.161747,0.190590,1.652720
LaserShip,296.0,0.184041,0.206184,0.012293,0.135054,0.151411,0.184717,3.379099
OnTrac,295.0,0.173136,0.085505,0.012395,0.135542,0.148712,0.179254,1.075069
UPS,251.0,0.186957,0.162009,0.117416,0.140099,0.156746,0.186137,2.501655
USPS,281.0,0.159827,0.069115,0.108254,0.128491,0.144197,0.166800,0.899667


In [39]:
df_raw["Carrier_Median_CPM"] = (
    df_raw.groupby("Carrier")["Cost_per_Mile"].transform("median")
)

In [40]:
df_raw.loc[df_raw["Cost"].isna(), "Cost"] = (
    df_raw.loc[df_raw["Cost"].isna(), "Distance_miles"] *
    df_raw.loc[df_raw["Cost"].isna(), "Carrier_Median_CPM"]
)

In [41]:
df_raw["Cost"].isna().sum()

np.int64(0)

In [42]:
df_raw["Cost"].describe()

count    2000.000000
mean      204.536370
std       220.864973
min        14.563905
25%       117.725000
50%       196.065000
75%       271.865000
max      6562.210000
Name: Cost, dtype: float64

In [43]:
df_raw.isnull().sum()

Shipment_ID                 0
Origin_Warehouse            0
Destination                 0
Carrier                     0
Shipment_Date               0
Delivery_Date              32
Weight_kg                   0
Cost                        0
Status                      0
Distance_miles              0
Transit_Days                0
Estimated_Delivery_Date     0
Cost_per_Mile              41
Carrier_Median_CPM          0
dtype: int64

In [44]:
df_raw["Cost_per_Mile"] = df_raw["Cost"] / df_raw["Distance_miles"]

In [45]:
df_raw["Cost_per_Mile"].isnull().sum()

np.int64(0)

In [46]:
df_raw.isnull().sum()

Shipment_ID                 0
Origin_Warehouse            0
Destination                 0
Carrier                     0
Shipment_Date               0
Delivery_Date              32
Weight_kg                   0
Cost                        0
Status                      0
Distance_miles              0
Transit_Days                0
Estimated_Delivery_Date     0
Cost_per_Mile               0
Carrier_Median_CPM          0
dtype: int64

In [48]:
df_raw["Delay_Days"] = (
    df_raw["Delivery_Date"] - df_raw["Estimated_Delivery_Date"]
).dt.days

In [49]:
df_raw["On_Time"] = df_raw["Delay_Days"] <= 0

In [51]:
df_raw.groupby("Carrier")["On_Time"].mean()

Carrier
Amazon Logistics    0.868613
DHL                 0.836299
FedEx               0.915254
LaserShip           0.897690
OnTrac              0.889632
UPS                 0.878906
USPS                0.921233
Name: On_Time, dtype: float64

In [52]:
df_delivered = df_raw[df_raw["Status"] == "Delivered"].copy()

df_delivered.groupby("Carrier")["On_Time"].mean()

Carrier
Amazon Logistics    0.995392
DHL                 0.974026
FedEx               0.995902
LaserShip           0.979839
OnTrac              0.983871
UPS                 0.968326
USPS                0.987448
Name: On_Time, dtype: float64

In [53]:
carrier_summary = df_delivered.groupby("Carrier").agg(
    Shipment_Count=("Shipment_ID", "count"),
    On_Time_Pct=("On_Time", "mean"),
    Avg_Cost=("Cost", "mean"),
    Total_Spend=("Cost", "sum"),
    Median_CPM=("Cost_per_Mile", "median")
).reset_index()

carrier_summary.sort_values("Total_Spend", ascending=False)

,Carrier,Shipment_Count,On_Time_Pct,Avg_Cost,Total_Spend,Median_CPM
2,FedEx,244,0.995902,220.287500,53750.150069,0.160878
5,UPS,221,0.968326,240.118601,53066.210742,0.155106
3,LaserShip,248,0.979839,208.642760,51743.404458,0.151411
1,DHL,231,0.974026,223.574342,51645.672926,0.169636
4,OnTrac,248,0.983871,183.410589,45485.826173,0.148712
6,USPS,239,0.987448,182.439323,43602.998139,0.144062
0,Amazon Logistics,217,0.995392,197.794171,42921.335025,0.147827


In [55]:
carrier_summary["Cost_Level"] = np.where( carrier_summary["Avg_Cost"] > carrier_summary["Avg_Cost"].mean(), "High Cost", "Low Cost" )

In [56]:
carrier_summary["Reliability_Level"] = np.where( carrier_summary["On_Time_Pct"] > carrier_summary["On_Time_Pct"].mean(), "High Reliability", "Lower Reliability" )

In [57]:
carrier_summary

,Carrier,Shipment_Count,On_Time_Pct,Avg_Cost,Total_Spend,Median_CPM,Cost_Level,Reliability_Level
0,Amazon Logistics,217,0.995392,197.794171,42921.335025,0.147827,Low Cost,High Reliability
1,DHL,231,0.974026,223.574342,51645.672926,0.169636,High Cost,Lower Reliability
2,FedEx,244,0.995902,220.287500,53750.150069,0.160878,High Cost,High Reliability
3,LaserShip,248,0.979839,208.642760,51743.404458,0.151411,High Cost,Lower Reliability
4,OnTrac,248,0.983871,183.410589,45485.826173,0.148712,Low Cost,High Reliability
5,UPS,221,0.968326,240.118601,53066.210742,0.155106,High Cost,Lower Reliability
6,USPS,239,0.987448,182.439323,43602.998139,0.144062,Low Cost,High Reliability


In [60]:
df_delivered["Distance_Bucket"] = pd.cut(
    df_delivered["Distance_miles"],
    bins=[0,300,600,900,float("inf")],
    labels=["0-300","301-600","601-900","900+"]
)

In [61]:
bucket_summary = df_delivered.groupby(
    ["Distance_Bucket", "Carrier"]
).agg(
    Shipments=("Shipment_ID", "count"),
    Avg_Cost=("Cost", "mean"),
    On_Time_Pct=("On_Time", "mean"),
    Median_CPM=("Cost_per_Mile", "median")
).reset_index()

bucket_summary.sort_values(["Distance_Bucket", "Shipments"], ascending=[True, False])

C:\Users\utkar\AppData\Local\Temp\ipykernel_11252\915303745.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_summary = df_delivered.groupby(


,Distance_Bucket,Carrier,Shipments,Avg_Cost,On_Time_Pct,Median_CPM
3,0-300,LaserShip,30,58.372376,1.000000,0.248230
1,0-300,DHL,24,54.912447,1.000000,0.256800
0,0-300,Amazon Logistics,23,67.553913,1.000000,0.308198
4,0-300,OnTrac,23,63.630000,0.956522,0.225393
6,0-300,USPS,19,42.133890,1.000000,0.191745
2,0-300,FedEx,17,68.008235,0.941176,0.361093
5,0-300,UPS,16,62.138750,0.937500,0.277150
11,301-600,OnTrac,38,90.854737,0.973684,0.197555
13,301-600,USPS,34,78.575047,0.970588,0.156095
10,301-600,LaserShip,29,90.581724,0.896552,0.198032


In [62]:
df_delivered.to_csv("carrier_optimization_analysis.csv", index=False)

In [64]:
df_delivered.to_csv(r"C:\Users\utkar\OneDrive\Desktop\carrier_optimization_analysis.csv", index=False)